# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Szan-12345/FLYRANK-MACHINE-LEARNING/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane and why

**Lane: Refresh / Content Opportunity Scoring.**

Picking this because I've already seen its shape work end-to-end — the starter pipeline in this
repo (`scripts/01`–`05`) builds exactly this: a ranked queue of pages to review, from observed
signals, validated on held-out clients. Week 1 also surfaced two things that make this lane worth
digging into rather than a default pick:

- The obvious heuristic — "old content is what's declining" — doesn't hold up (see section 3).
  A hand rule isn't enough here; there's a real pattern to find, not just common sense to encode.
- A learned model already beat a hand-written rule by roughly 3x on Precision@50 on this starter
  slice — evidence the signal is real and learnable, not noise.

CTR/Engagement scoring fit my Week 1 work just as well, but "which page do I look at first" is
the more general decision, and it's the one this repo's reference pipeline is already built
around — a stronger foundation for the next 7 weeks.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import json
import os

# --- AUTOMATIC PATH LOCATOR ---
# Walk up or down to find the directory containing 'data'
for i in range(3):
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    elif os.path.exists("FLYRANK-MACHINE-LEARNING-main/data/raw/content_refresh_anonymized.csv"):
        os.chdir("FLYRANK-MACHINE-LEARNING-main")
        break
    else:
        os.chdir("..") 

# Load the dataset safely
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# ------------------------------

# Now run your value counts check
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. The question

**Unit of analysis:** one page (`content_id`) — trailing-90-day metrics, one row per page.

**The decision:** each cycle, a content team has capacity to manually review only some of the
inventory. Which pages make that shortlist?

**Who acts, and what they do:** a content strategist opens a flagged page, checks its reason
codes, and picks a specific action — rewrite, expand thin sections, update stale facts, fix a
technical issue, or leave it alone. The model doesn't choose the fix; it chooses where to look first.

**Cost of a wrong call:**
- False positive (flagged, not actually worth it) → wastes limited reviewer hours, and because
  capacity is fixed, that's a genuinely declining page losing its slot.
- False negative (a real decliner never flagged) → traffic keeps eroding silently, unreviewed,
  and the miss compounds the longer it goes unnoticed.
Both costs are about the ranking's *order*, which is why Precision@K — not raw accuracy — is the
right metric.

**Why data/ML, not just a rule:** section 3 shows the obvious rule (target old content) doesn't
match what's actually declining. A plain if-statement misses this; a model weighing several
signals together doesn't.

**One-paragraph frame:**
> For a content team with limited review capacity, deciding which pages to look at first this
> cycle, we will build a ranked priority list from trailing-90-day signals, scoring decline risk,
> measured by Precision@K against pages that actually kept declining. A wrong call costs wasted
> reviewer hours or a missed real decline. A plain age-based rule isn't enough because age and
> decline don't line up in this data. We claim only decision-support, observed-pattern results —
> not that reviewing a flagged page will fix it.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(f"Total pages in this starter slice: {len(df):,}")
print("Even reviewing 50 pages a week, a team can't manually check all of these —")
print("which is why ranking matters, not just flagging.")

Total pages in this starter slice: 30,000
Even reviewing 50 pages a week, a team can't manually check all of these —
which is why ranking matters, not just flagging.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import json
import os

# --- AUTOMATIC PATH LOCATOR ---
# Walk up or down to find the directory containing 'data' and 'outputs'
for i in range(3):
    if os.path.exists("data/raw/content_refresh_anonymized.csv"):
        break
    elif os.path.exists("FLYRANK-MACHINE-LEARNING-main/data/raw/content_refresh_anonymized.csv"):
        os.chdir("FLYRANK-MACHINE-LEARNING-main")
        break
    else:
        os.chdir("..") # Move up one folder level and check again

print(f"Current working directory locked to: {os.getcwd()}")
# ------------------------------

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Does the obvious heuristic (old content declines) hold up?
age_by_trend = df.groupby("trend_direction")["content_age_days"].median()
print("\nMedian content age (days) by trend:")
print(age_by_trend.round(0).to_string())

# Is there learnable signal beyond a hand rule?
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]
print(f"\nBaseline Precision@50: {base:.3f}  |  Random forest: {rf:.3f}  |  Lift: {rf/base:.1f}x")

# How big is the review problem?
down_n = (df["trend_direction"] == "down").sum()
print(f"\n'down' pages: {down_n} of {len(df)} ({down_n/len(df)*100:.1f}%)")

Current working directory locked to: c:\Users\Zohaib Ali\Downloads\FLYRANK-MACHINE-LEARNING-main\FLYRANK-MACHINE-LEARNING-main

Median content age (days) by trend:
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        292.0

Baseline Precision@50: 0.240  |  Random forest: 0.680  |  Lift: 2.8x

'down' pages: 16262 of 30000 (54.2%)


## 4. Careful words — what I can and can't claim

**I can claim:** these patterns are observed and directional in this 30,000-row anonymized
starter slice, validated with client-holdout splits. A learned ranking beats a fixed rule by a
wide enough margin (~3x) that it's unlikely to be noise. The age-vs-decline mismatch is real in
this sample and worth investigating further.

**I can't claim:** that reviewing a flagged page *causes* recovery — that needs an experiment, not
this observational data. That I've proven anything about how a search algorithm works — this is
FlyRank's own observed traffic signals, not search engine internals. That the 54% "down" rate is
FlyRank's real-world decline rate — this is a teaching sample, not a representative draw from the
~520,000-item warehouse, so I'm using it to illustrate scale, not as a true base rate. That
today's `trend_direction` is the label I should train the real model on — it's a proxy computed
from the current window, not a future outcome; the stronger version of this project defines the
label as prior-90-days features predicting a next-30-days outcome. That every "down" page is a
genuine decline — some could be seasonal or consolidating onto a sibling page, and I haven't ruled
that out yet (next session's signal audit).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
low_volume_down = df[(df["trend_direction"]=="down") & (df["impressions_90d"] < 100)]
print(f"Of {len(df[df['trend_direction']=='down']):,} 'down' pages, {len(low_volume_down):,} have")
print("under 100 impressions/90d — too little traffic to confidently call 'declining' vs. just noisy.")

Of 16,262 'down' pages, 3,110 have
under 100 impressions/90d — too little traffic to confidently call 'declining' vs. just noisy.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.